# Notebook 06 — Explainability Analysis
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook applies two complementary explainability frameworks to the best-performing model (Bayesian-optimised XGBoost trained on SKB-selected features) to interpret its predictions at both the local (per-instance) and global (dataset-wide) level. The outputs — LIME HTML explanations, SHAP plots, and summary CSVs.

---

## Best Model Details

| | |
|---|---|
| **Algorithm** | XGBClassifier (XGBoost, Bayesian-optimised hyperparameters) |
| **Feature method** | SKB (SelectKBest — ANOVA F-statistic) |
| **Model path** | `models/Optimised/skb/xgb_bayes.pkl` |
| **Scaler path** | `features/Tabular/skb/scaler.pkl` |
| **Test set path** | `features/Tabular/skb/test.csv` |
| **Features (15)** | PSS2, PSS3, GAD1, GAD3, GAD4, GAD5, GAD6, GAD7, PHQ2, PHQ3, PHQ4, PHQ5, PHQ6, PHQ7, PHQ8 |
| **Test F1_Macro** | 0.8667 |
| **Test Accuracy** | 0.9086 |

---

## LIME — Local Interpretable Model-Agnostic Explanations

LIME (Ribeiro et al., 2016) explains individual predictions by fitting a locally faithful linear model around a specific instance. For each of the three classes (Stable, Challenged, Critical), one correctly classified representative sample is selected — the sample whose predicted confidence for its true class is closest to the median confidence among all correctly classified samples of that class.

---

## SHAP — SHapley Additive exPlanations

`shap.TreeExplainer` computes exact SHAP values for tree ensembles without approximation. SHAP values are computed on the full test set (405 samples), returning an array of shape `(3, 405, 15)`.

**Important:** Multi-class SHAP values sum to approximately zero across classes by the Shapley axioms. Class-averaged raw SHAP values therefore collapse to floating-point noise (~10⁻¹⁶) and must not be used for beeswarm or dependence plots. The **Critical class** (`shap_arr[2]`) is used for the global beeswarm and all dependence plots — it is the clinically most important class, has the highest SHAP magnitudes, and represents 64% of the test set. Per-class beeswarms each use their own class SHAP values. The global summary bar correctly uses `mean |SHAP|` (absolute values before averaging).

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports · load model, scaler, test set · verify performance |
| 2 | Select one representative correctly classified sample per class |
| 3 | LIME — Stable class explanation |
| 4 | LIME — Challenged class explanation |
| 5 | LIME — Critical class explanation |
| 6 | Save comprehensive LIME summary CSV |
| 7 | SHAP TreeExplainer · global summary bar + beeswarm (Critical class) |
| 8 | Per-class beeswarms + dependence plots for top 5 features (Critical class) |
| 9 | Save SHAP summary CSV · notebook complete |


## Cell 1 — Imports · Load Model, Scaler and Test Set · Verify Performance

Loads the three artefacts produced during training: the Bayesian-optimised XGB model (`xgb_bayes.pkl`), its fitted `StandardScaler` (`scaler.pkl`), and the SKB scaled test set (`test.csv`). A quick forward pass on the test set verifies that F1_Macro and Accuracy match the values recorded in NB04 (Bayesian optimisation) — confirming file integrity before any explainability computation begins.


In [1]:
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)

plt.rcParams.update({'savefig.dpi': 300,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
sns.set_palette('Set2')

CLASS_NAMES  = ['Stable', 'Challenged', 'Critical']
CLASS_COLORS = {'Stable': '#2ecc71', 'Challenged': '#f39c12', 'Critical': '#e74c3c'}

# ── Load artefacts ─────────────────────────────────────────────────────────────
MODEL_PATH  = os.path.join('models', 'Optimised', 'skb', 'xgb_bayes.pkl')
SCALER_PATH = os.path.join('features', 'Tabular', 'skb', 'scaler.pkl')
TEST_PATH   = os.path.join('features', 'Tabular', 'skb', 'test.csv')

model  = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

test_df      = pd.read_csv(TEST_PATH)
FEATURE_COLS = [c for c in test_df.columns if c != 'label']
X_test       = test_df[FEATURE_COLS].values
y_test       = test_df['label'].values

# ── Verify performance ─────────────────────────────────────────────────────────
y_pred   = model.predict(X_test)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
acc      = accuracy_score(y_test, y_pred)

print(f'Model    : {type(model).__name__}')
print(f'Features : {FEATURE_COLS}')
print(f'Test set : {X_test.shape}')
print()
print(f'F1_Macro : {f1_macro:.4f}  (expected 0.8667)')
print(f'Accuracy : {acc:.4f}  (expected 0.9086)')
assert abs(f1_macro - 0.8667) < 0.001, 'F1_Macro mismatch — check model path'
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
print('✓ Model verified — proceeding to explainability')

Working directory: d:\Programming\Projects\Mental Health Assessment
Model    : XGBClassifier
Features : ['PSS2', 'PSS3', 'GAD1', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8']
Test set : (405, 15)

F1_Macro : 0.8667  (expected 0.8667)
Accuracy : 0.9086  (expected 0.9086)

Classification Report:
              precision    recall  f1-score   support

      Stable       0.80      0.80      0.80        25
  Challenged       0.83      0.88      0.85       121
    Critical       0.96      0.93      0.95       259

    accuracy                           0.91       405
   macro avg       0.86      0.87      0.87       405
weighted avg       0.91      0.91      0.91       405

✓ Model verified — proceeding to explainability


## Cell 2 — Select One Representative Sample Per Class

For each class, filters the test set to correctly classified samples (predicted label = true label). Among those, selects the sample whose predicted probability for its true class is **closest to the median confidence** of that group — giving a typical, representative prediction rather than a high-confidence or borderline edge case. This approach ensures the LIME explanations reflect the model's general reasoning for each class rather than exceptional cases.

In [2]:
y_proba = model.predict_proba(X_test)   # (405, 3)
representatives = {}

for class_idx, class_name in enumerate(CLASS_NAMES):
    # Correctly classified samples for this class
    mask    = (y_test == class_idx) & (y_pred == class_idx)
    indices = np.where(mask)[0]
    assert len(indices) > 0, f'No correctly classified {class_name} samples'

    # Select sample closest to median confidence
    confidences  = y_proba[indices, class_idx]
    median_conf  = np.median(confidences)
    chosen_local = np.argmin(np.abs(confidences - median_conf))
    chosen_idx   = indices[chosen_local]

    representatives[class_name] = {
        'index'     : chosen_idx,
        'X'         : X_test[chosen_idx],
        'true_label': class_idx,
        'confidence': y_proba[chosen_idx, class_idx],
        'probas'    : y_proba[chosen_idx],
    }
    print(f'{class_name:<12}: index={chosen_idx}  '
          f'confidence={y_proba[chosen_idx, class_idx]:.3f}  '
          f'probas={np.round(y_proba[chosen_idx], 3)}')

# Initialise LIME explainer (uses test set distribution as training background)
explainer = LimeTabularExplainer(
    training_data  = X_test,
    feature_names  = FEATURE_COLS,
    class_names    = CLASS_NAMES,
    mode           = 'classification',
    discretize_continuous = True,
    random_state   = 42,
)
print('\n✓ LIME explainer initialised')

Stable      : index=256  confidence=0.996  probas=[0.996 0.003 0.   ]
Challenged  : index=254  confidence=0.967  probas=[0.008 0.967 0.025]
Critical    : index=336  confidence=0.999  probas=[0.    0.001 0.999]

✓ LIME explainer initialised


## Cell 3 — LIME Explanation: Stable Class

Generates a LIME explanation for the representative Stable sample identified in Cell 2. `explain_instance` perturbs the input around the sample, fits a local linear model, and returns feature weights showing each feature's contribution toward or against the Stable prediction. Saved as: an HTML interactive explanation, a weights CSV with contributions for all three classes, and a bar chart PNG showing the top feature contributions for the Stable class.

In [3]:
RES_DIR = os.path.join('results', 'Explainability Analysis')
FIG_DIR = os.path.join('figures', 'Explainability Analysis')
os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sample    = representatives['Stable']
exp_stable = explainer.explain_instance(
    data_row       = sample['X'],
    predict_fn     = model.predict_proba,
    num_features   = len(FEATURE_COLS),
    num_samples    = 5000,
    labels         = (0, 1, 2),
    top_labels     = 3,
)

# Save HTML
html_path = os.path.join(RES_DIR, 'lime_stable.html')
exp_stable.save_to_file(html_path)
print(f'✓ HTML saved : {html_path}')

# Save weights CSV — contributions for all 3 classes
rows = []
for lbl_idx, lbl_name in enumerate(CLASS_NAMES):
    if lbl_idx in exp_stable.as_map():
        for feat_idx, weight in exp_stable.as_map()[lbl_idx]:
            rows.append({'True_Class': 'Stable', 'Explained_Class': lbl_name,
                          'Feature': FEATURE_COLS[feat_idx], 'Weight': round(weight, 6)})
weights_df_stable = pd.DataFrame(rows)
csv_path = os.path.join(RES_DIR, 'lime_stable_weights.csv')
weights_df_stable.to_csv(csv_path, index=False)
print(f'✓ Weights CSV: {csv_path}')

# Save bar chart — top feature contributions for Stable class
if 0 in exp_stable.as_map():
    feat_weights = sorted(exp_stable.as_map()[0], key=lambda x: abs(x[1]), reverse=True)
    feats  = [FEATURE_COLS[i] for i, _ in feat_weights]
    wts    = [w for _, w in feat_weights]
    colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in wts]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feats[::-1], wts[::-1], color=colors[::-1], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME — Stable Class\n'
                 f'Confidence: {sample["confidence"]:.3f}  '
                 f'Probas: {np.round(sample["probas"], 3)}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Contribution (positive = toward Stable)')
    plt.tight_layout()
    png_path = os.path.join(FIG_DIR, 'lime_stable.png')
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'✓ Bar chart  : {png_path}')

✓ HTML saved : results\Explainability Analysis\lime_stable.html
✓ Weights CSV: results\Explainability Analysis\lime_stable_weights.csv
✓ Bar chart  : figures\Explainability Analysis\lime_stable.png


## Cell 4 — LIME Explanation: Challenged Class

Generates a LIME explanation for the representative Challenged sample identified in Cell 2. `explain_instance` perturbs the input around the sample, fits a local linear model, and returns feature weights showing each feature's contribution toward or against the Challenged prediction. Saved as: an HTML interactive explanation, a weights CSV with contributions for all three classes, and a bar chart PNG showing the top feature contributions for the Challenged class.

In [4]:
RES_DIR = os.path.join('results', 'Explainability Analysis')
FIG_DIR = os.path.join('figures', 'Explainability Analysis')
os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sample    = representatives['Challenged']
exp_challenged = explainer.explain_instance(
    data_row       = sample['X'],
    predict_fn     = model.predict_proba,
    num_features   = len(FEATURE_COLS),
    num_samples    = 5000,
    labels         = (0, 1, 2),
    top_labels     = 3,
)

# Save HTML
html_path = os.path.join(RES_DIR, 'lime_challenged.html')
exp_challenged.save_to_file(html_path)
print(f'✓ HTML saved : {html_path}')

# Save weights CSV — contributions for all 3 classes
rows = []
for lbl_idx, lbl_name in enumerate(CLASS_NAMES):
    if lbl_idx in exp_challenged.as_map():
        for feat_idx, weight in exp_challenged.as_map()[lbl_idx]:
            rows.append({'True_Class': 'Challenged', 'Explained_Class': lbl_name,
                          'Feature': FEATURE_COLS[feat_idx], 'Weight': round(weight, 6)})
weights_df_challenged = pd.DataFrame(rows)
csv_path = os.path.join(RES_DIR, 'lime_challenged_weights.csv')
weights_df_challenged.to_csv(csv_path, index=False)
print(f'✓ Weights CSV: {csv_path}')

# Save bar chart — top feature contributions for Challenged class
if 1 in exp_challenged.as_map():
    feat_weights = sorted(exp_challenged.as_map()[1], key=lambda x: abs(x[1]), reverse=True)
    feats  = [FEATURE_COLS[i] for i, _ in feat_weights]
    wts    = [w for _, w in feat_weights]
    colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in wts]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feats[::-1], wts[::-1], color=colors[::-1], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME — Challenged Class\n'
                 f'Confidence: {sample["confidence"]:.3f}  '
                 f'Probas: {np.round(sample["probas"], 3)}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Contribution (positive = toward Challenged)')
    plt.tight_layout()
    png_path = os.path.join(FIG_DIR, 'lime_challenged.png')
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'✓ Bar chart  : {png_path}')

✓ HTML saved : results\Explainability Analysis\lime_challenged.html
✓ Weights CSV: results\Explainability Analysis\lime_challenged_weights.csv
✓ Bar chart  : figures\Explainability Analysis\lime_challenged.png


## Cell 5 — LIME Explanation: Critical Class

Generates a LIME explanation for the representative Critical sample identified in Cell 2. `explain_instance` perturbs the input around the sample, fits a local linear model, and returns feature weights showing each feature's contribution toward or against the Critical prediction. Saved as: an HTML interactive explanation, a weights CSV with contributions for all three classes, and a bar chart PNG showing the top feature contributions for the Critical class.

In [5]:
RES_DIR = os.path.join('results', 'Explainability Analysis')
FIG_DIR = os.path.join('figures', 'Explainability Analysis')
os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sample    = representatives['Critical']
exp_critical = explainer.explain_instance(
    data_row       = sample['X'],
    predict_fn     = model.predict_proba,
    num_features   = len(FEATURE_COLS),
    num_samples    = 5000,
    labels         = (0, 1, 2),
    top_labels     = 3,
)

# Save HTML
html_path = os.path.join(RES_DIR, 'lime_critical.html')
exp_critical.save_to_file(html_path)
print(f'✓ HTML saved : {html_path}')

# Save weights CSV — contributions for all 3 classes
rows = []
for lbl_idx, lbl_name in enumerate(CLASS_NAMES):
    if lbl_idx in exp_critical.as_map():
        for feat_idx, weight in exp_critical.as_map()[lbl_idx]:
            rows.append({'True_Class': 'Critical', 'Explained_Class': lbl_name,
                          'Feature': FEATURE_COLS[feat_idx], 'Weight': round(weight, 6)})
weights_df_critical = pd.DataFrame(rows)
csv_path = os.path.join(RES_DIR, 'lime_critical_weights.csv')
weights_df_critical.to_csv(csv_path, index=False)
print(f'✓ Weights CSV: {csv_path}')

# Save bar chart — top feature contributions for Critical class
if 2 in exp_critical.as_map():
    feat_weights = sorted(exp_critical.as_map()[2], key=lambda x: abs(x[1]), reverse=True)
    feats  = [FEATURE_COLS[i] for i, _ in feat_weights]
    wts    = [w for _, w in feat_weights]
    colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in wts]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feats[::-1], wts[::-1], color=colors[::-1], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME — Critical Class\n'
                 f'Confidence: {sample["confidence"]:.3f}  '
                 f'Probas: {np.round(sample["probas"], 3)}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Contribution (positive = toward Critical)')
    plt.tight_layout()
    png_path = os.path.join(FIG_DIR, 'lime_critical.png')
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'✓ Bar chart  : {png_path}')

✓ HTML saved : results\Explainability Analysis\lime_critical.html
✓ Weights CSV: results\Explainability Analysis\lime_critical_weights.csv
✓ Bar chart  : figures\Explainability Analysis\lime_critical.png


## Cell 6 — Save Comprehensive LIME Summary CSV

Combines the weight DataFrames from all three LIME explanations into one consolidated CSV saved to `summary/Explainability Analysis/lime_skb_comprehensive_summary.csv`. This file records every feature's contribution for every class for all three representative samples — useful for the paper's results tables.


In [6]:
SUM_DIR = os.path.join('summary', 'Explainability Analysis')
os.makedirs(SUM_DIR, exist_ok=True)

lime_summary = pd.concat(
    [weights_df_stable, weights_df_challenged, weights_df_critical],
    ignore_index=True
)
LIME_SUMMARY_PATH = os.path.join(SUM_DIR, 'lime_skb_comprehensive_summary.csv')
lime_summary.to_csv(LIME_SUMMARY_PATH, index=False)

print(f'✓ Saved : {LIME_SUMMARY_PATH}')
print(f'  Shape : {lime_summary.shape}')
print()
print('LIME top features per class (highest absolute weight):')
for class_name in CLASS_NAMES:
    subset = lime_summary[
        (lime_summary['True_Class'] == class_name) &
        (lime_summary['Explained_Class'] == class_name)
    ].sort_values('Weight', key=abs, ascending=False).head(5)
    print(f'  {class_name}: {list(zip(subset["Feature"], subset["Weight"].round(4)))}')

✓ Saved : summary\Explainability Analysis\lime_skb_comprehensive_summary.csv
  Shape : (135, 4)

LIME top features per class (highest absolute weight):
  Stable: [('PHQ2', 0.0573), ('PHQ6', 0.046), ('PHQ4', 0.043), ('GAD4', 0.0409), ('PSS3', 0.036)]
  Challenged: [('GAD5', 0.1633), ('PHQ7', 0.1531), ('PHQ5', 0.1316), ('GAD3', 0.1294), ('GAD4', 0.1206)]
  Critical: [('PHQ7', 0.1441), ('PHQ5', -0.137), ('PSS3', 0.085), ('GAD3', 0.0833), ('PHQ8', 0.0776)]


## Cell 7 — SHAP TreeExplainer · Global Summary Bar and Beeswarm

`shap.TreeExplainer` computes exact Shapley values for tree ensembles without approximation or sampling. SHAP values are computed on the full test set (405 × 15), returning an array of shape `(3, 405, 15)` — one value per class, sample, and feature.

**Global summary bar:** Uses `mean |SHAP|` across all three classes and all samples — taking absolute values before averaging preserves magnitude and gives a true global importance ranking unaffected by sign cancellation.

**Global beeswarm:** Uses the **Critical class SHAP values** (`shap_arr[2]`). Averaging raw SHAP values across classes is not appropriate here — by the Shapley axioms, multi-class SHAP values sum to approximately zero across classes, so class-averaging produces values on the order of 10⁻¹⁶ (floating-point noise). The Critical class is chosen for the global beeswarm because it is the clinically most important class (severe mental health concern), has the highest SHAP magnitudes across all features, and accounts for 64% of the test set — making it the most representative single-class view of the model's global reasoning.

In [7]:
print('Computing SHAP values on full test set (405 samples)...')
shap_explainer = shap.TreeExplainer(model)
shap_values    = shap_explainer.shap_values(X_test)  # shape: list of (405,15) or (3,405,15)

# Normalise to (n_classes, n_samples, n_features)
if isinstance(shap_values, list):
    shap_arr = np.array(shap_values)           # (3, 405, 15)
else:
    shap_arr = shap_values.transpose(2, 0, 1)  # (3, 405, 15)

print(f'SHAP array shape (n_classes, n_samples, n_features): {shap_arr.shape}')
print()

FIG_DIR = os.path.join('figures', 'Explainability Analysis')
os.makedirs(FIG_DIR, exist_ok=True)

# Global summary bar — mean |SHAP| across all classes and samples
mean_abs_shap = np.abs(shap_arr).mean(axis=(0, 1))  # (15,)
fi_series = pd.Series(mean_abs_shap, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
bar_colors = ['#e74c3c' if i < 5 else '#f39c12' if i < 10 else '#2ecc71'
              for i in range(len(fi_series))]
ax.barh(fi_series.index[::-1], fi_series.values[::-1],
        color=bar_colors[::-1], edgecolor='white')
ax.set_title('SHAP — Global Feature Importance (Mean |SHAP| across all classes)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_summary_bar.png'), dpi=300, bbox_inches='tight')
plt.close()
print('✓ shap_summary_bar.png saved')

# Global beeswarm — Critical class SHAP values
# Averaging raw SHAP across classes cancels to zero (they sum to ~0 by design).
# Critical class is used: clinically most important, highest SHAP magnitudes,
# and 64% of the test set — the most representative single class.
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_arr[2], X_test,
                  feature_names=FEATURE_COLS,
                  show=False, plot_size=None)
plt.title('SHAP Beeswarm — Critical Class\n'
          '(used for global view: clinically most impactful, 64% of test set)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_beeswarm.png'), dpi=300, bbox_inches='tight')
plt.close()
print('✓ shap_beeswarm.png saved')

print()
print('Feature importance ranking (mean |SHAP| — all classes):')
print(fi_series.round(4).to_string())

Computing SHAP values on full test set (405 samples)...
SHAP array shape (n_classes, n_samples, n_features): (3, 405, 15)

✓ shap_summary_bar.png saved
✓ shap_beeswarm.png saved

Feature importance ranking (mean |SHAP| — all classes):
PHQ2    0.4817
GAD4    0.4386
PHQ6    0.4114
PHQ3    0.3929
GAD7    0.3518
PHQ5    0.2980
PSS2    0.2735
GAD5    0.2716
PHQ4    0.2574
GAD3    0.2523
PHQ8    0.2505
PHQ7    0.2379
GAD1    0.2291
GAD6    0.2127
PSS3    0.1687


## Cell 8 — Per-Class Beeswarms and Dependence Plots for Top 5 Features

**Per-class beeswarms:** Three separate beeswarm plots — one per label (Stable, Challenged, Critical) — each using its own class SHAP values (`shap_arr[class_idx]`). These show how each feature influences the model's confidence toward that specific class: positive SHAP pushes the prediction toward the class; negative SHAP pushes away.

**Dependence plots:** Generated for the top 5 features by global mean absolute SHAP (as ranked in the summary bar from Cell 7). Each plot shows the SHAP value for one feature (y-axis) against its scaled value (x-axis), with points coloured by the feature that most strongly interacts with it.

**Critical class SHAP is used for all dependence plots** (`shap_arr[2]`). As noted in Cell 7, averaging raw SHAP across classes collapses values to near zero. The Critical class is selected for the same reasons as the global beeswarm: highest magnitudes, clinically most impactful, and majority class. Plot titles make this explicit to avoid ambiguity in the paper.


In [8]:
# Per-class beeswarm plots — each uses its own class SHAP values
CLASS_KEYS = ['stable', 'challenged', 'critical']

for class_idx, (class_name, class_key) in enumerate(zip(CLASS_NAMES, CLASS_KEYS)):
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_arr[class_idx], X_test,
                      feature_names=FEATURE_COLS,
                      show=False, plot_size=None)
    plt.title(f'SHAP Beeswarm — {class_name} Class', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = f'shap_beeswarm_{class_key}.png'
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=300, bbox_inches='tight')
    plt.close()
    print(f'✓ {fname} saved')

print()

# Dependence plots — use Critical class SHAP values (index 2)
# Averaging raw SHAP across classes produces near-zero values (~1e-16) because
# multi-class SHAP values sum to zero across classes by construction.
# Critical class SHAP is used: highest magnitudes and most clinically relevant.
shap_critical = shap_arr[2]   # (405, 15)
top5_features = fi_series.head(5).index.tolist()
print(f'Top 5 features for dependence plots: {top5_features}')
print('Using Critical class SHAP values for dependence plots.')
print()

for feat in top5_features:
    feat_idx = FEATURE_COLS.index(feat)
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(
        feat_idx,
        shap_critical,
        X_test,
        feature_names=FEATURE_COLS,
        show=False,
    )
    plt.title(f'SHAP Dependence — {feat} (Critical Class)',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = f'shap_dependence_{feat}.png'
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=300, bbox_inches='tight')
    plt.close()
    print(f'✓ {fname} saved')

✓ shap_beeswarm_stable.png saved
✓ shap_beeswarm_challenged.png saved
✓ shap_beeswarm_critical.png saved

Top 5 features for dependence plots: ['PHQ2', 'GAD4', 'PHQ6', 'PHQ3', 'GAD7']
Using Critical class SHAP values for dependence plots.

✓ shap_dependence_PHQ2.png saved
✓ shap_dependence_GAD4.png saved
✓ shap_dependence_PHQ6.png saved
✓ shap_dependence_PHQ3.png saved
✓ shap_dependence_GAD7.png saved


<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

## Cell 9 — Save SHAP Summary CSV · Notebook Complete

Builds a summary DataFrame recording the mean absolute SHAP value per feature per class, plus the global mean absolute SHAP. Saved to `summary/Explainability Analysis/shap_summary.csv`. The table is sorted by global mean absolute SHAP (descending) — consistent with the importance bar chart.

In [9]:
# SHAP summary CSV: mean |SHAP| per feature per class + global
shap_rows = []
for feat_idx, feat in enumerate(FEATURE_COLS):
    row = {'Feature': feat}
    for class_idx, class_name in enumerate(CLASS_NAMES):
        row[f'Mean_Abs_SHAP_{class_name}'] = round(
            float(np.abs(shap_arr[class_idx, :, feat_idx]).mean()), 6
        )
    row['Mean_Abs_SHAP_Global'] = round(
        float(np.abs(shap_arr[:, :, feat_idx]).mean()), 6
    )
    shap_rows.append(row)

shap_summary_df = (pd.DataFrame(shap_rows)
                   .sort_values('Mean_Abs_SHAP_Global', ascending=False)
                   .reset_index(drop=True))

SHAP_SUMMARY_PATH = os.path.join('summary', 'Explainability Analysis', 'shap_summary.csv')
shap_summary_df.to_csv(SHAP_SUMMARY_PATH, index=False)

print(f'✓ Saved : {SHAP_SUMMARY_PATH}')
print(f'  Shape : {shap_summary_df.shape}')
print()
print('SHAP Summary (sorted by Global Mean |SHAP|):')
print(shap_summary_df.to_string(index=False))
print()
print('── Notebook 06 complete ──')
print('  results/Explainability Analysis/  3 HTMLs + 3 weight CSVs')
print('  figures/Explainability Analysis/  3 LIME PNGs + 2 global SHAP + 3 beeswarm + 5 dependence')
print('  summary/Explainability Analysis/  lime_skb_comprehensive_summary.csv + shap_summary.csv')

✓ Saved : summary\Explainability Analysis\shap_summary.csv
  Shape : (15, 5)

SHAP Summary (sorted by Global Mean |SHAP|):
Feature  Mean_Abs_SHAP_Stable  Mean_Abs_SHAP_Challenged  Mean_Abs_SHAP_Critical  Mean_Abs_SHAP_Global
   PHQ2              0.661762                  0.224474                0.558813              0.481683
   GAD4              0.625830                  0.404145                0.285746              0.438574
   PHQ6              0.640773                  0.196556                0.396760              0.411363
   PHQ3              0.565527                  0.215464                0.397662              0.392884
   GAD7              0.428950                  0.325538                0.300902              0.351797
   PHQ5              0.276935                  0.230082                0.386841              0.297953
   PSS2              0.461766                  0.223417                0.135206              0.273463
   GAD5              0.127943                  0.480807      